In [22]:
import requests
from pydantic import BaseModel
from typing import Dict, Any, Optional, List

class AllocatedResources(BaseModel):
    request_id: int
    resource_center_ids: List[int]
    quantities: List[int]

class UserMessage(BaseModel):
    message: str

class AgentState(BaseModel):
    agent: Optional[str] = "workflow"
    input: Optional[Dict[str, Any]] = None
    image_path: Optional[str] = None
    voice_path: Optional[str] = None
    request: Optional[Dict[str, Any]] = None
    image_description: Optional[str] = None
    voice_description: Optional[str] = None
    status: Optional[str] = "pending"
    reason: Optional[str] = None
    available_resources: Optional[List[Dict[str, Any]]] = None
    allocated_resources: Optional[AllocatedResources] = None
    disaster_status: Optional[str] = "PENDING"
    user_msg: Optional[UserMessage] = None
    error_msg: Optional[str] = None



PROMPT = f"""
    You are an intelligent resource assignment agent.
    Your task is to allocate the available resources from  to the disaster request.
         
    Definitions:
        - count = total resources at a center
        - used = already allocated resources
        - available = count - used
        - resourceId = resource center id

    Output format:
    {{
        "request_id": "<id>", id from state.request
        "resource_center_ids": [<list of resource center ids which can assign to this request>],
        "quantities": [<list of quantities corresponding to each resource center id>]
    }}

    Rules:
            1. Never allocate more than available (no over-allocation).
            2. Do not exhaust all resources at once (keep reserves).
            3. Consider disaster urgency when deciding quantities.
            4. Apply an optimal allocation strategy (balanced + fair).
            5. Prioritize centers by proximity and availability.
            6. Mark allocated quantities as used after assignment.
    """

schema = {
        "type": "object",
        "properties": {
            "request_id": {"type": "integer"},
            "resource_center_ids": {
                "type": "array",
                "items": {"type": "integer"}
            },
            "quantities": {
                "type": "array",
                "items": {"type": "integer"}
            }
        },
        "required": ["request_id", "resource_center_ids", "quantities"]
    }
res = requests.post(
        url="https://55713976f485.ngrok-free.app/api/generate",
        headers={"Content-Type": "application/json"},
        json={
                "model": "qwen3:4b",
                "prompt": PROMPT,
                "stream": False,
                "options": {"temperature": 0.2},
                "format": AllocatedResources.model_json_schema(),
                },
        )

print("Response",res.json()['response'])

Response {
  "request_id": 1,
  "resource_center_ids": [1, 2, 3],
  "quantities": [5, 5, 5]
}

